# So sanh optimizer trong tf.keras: GD / Momentum / Adam

Ban minh hoa dung **TensorFlow / Keras** song song voi bai 04 (tu viet
`update_parameters_with_gd`, `update_parameters_with_momentum`, `update_parameters_with_adam`,
`random_mini_batches` bang NumPy). Trong `tf.keras`, ca 3 thuat toan cap nhat trong so da co san
duoi dang **optimizer object**, va viec chia mini-batch duoc `model.fit(..., batch_size=64)` tu lo -
khong can tu viet `random_mini_batches`.

In [ ]:
# --- Google Colab setup: bo qua tu dong neu chay Jupyter local ---
import sys, os

if "google.colab" in sys.modules:
    REPO_DIR = "/content/AI-Teaching-Labs"
    if not os.path.isdir(REPO_DIR):
        !git clone --depth 1 https://github.com/NonomiyaIzumi/AI-Teaching-Labs.git {REPO_DIR}
    NOTEBOOK_DIR = f"{REPO_DIR}/Deep Learning/Module 1/keras-tensorflow/04_SGD-Momentum-Adam-Toi-uu-hoa"
    os.chdir(NOTEBOOK_DIR)
    print("Da chuyen working directory sang:", os.getcwd())
else:
    print("Khong phai Colab - dung working directory hien tai:", os.getcwd())
    print("Neu chay Jupyter local: tu thu muc keras-tensorflow/, chay 'uv sync' truoc.")

## 1. Du lieu & kien truc

Dung lai bo du lieu **Pima Indians Diabetes** va kien truc `[8, 5, 2, 1]` giong het bai 04 (NumPy).
`build_model()` (trong `keras_utils.py`) da dung san kien truc nay, khoi tao kieu He - vi bai nay
tap trung vao **optimizer**, khong phai kien truc/khoi tao (da hoc o bai 03).

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

from keras_utils import load_dataset, build_model, compile_and_train, evaluate_classification

tf.config.experimental.enable_op_determinism()  # dam bao ket qua giong het nhau giua cac lan chay

train_X, train_Y, test_X, test_Y = load_dataset()
print("train_X:", train_X.shape, " train_Y:", train_Y.shape)

## 2. Chon optimizer - Exercise 1

- `"gd"` -> Gradient Descent thuan, khong momentum, tuong duong `update_parameters_with_gd`
- `"momentum"` -> Gradient Descent + Momentum (`beta=0.9`), tuong duong `update_parameters_with_momentum`
- `"adam"` -> Adam (`beta1=0.9, beta2=0.999`), tuong duong `update_parameters_with_adam`

### Exercise 1 - `get_optimizer`

In [ ]:
def get_optimizer(name, learning_rate=0.0007):
    '''
    Tra ve mot optimizer Keras tuong ung voi 3 thuat toan da tu cai o bai 04.

    Arguments:
    name -- "gd" | "momentum" | "adam"

    Returns: mot tf.keras.optimizers.Optimizer
    '''
    # (~6 dong) dung keras.optimizers.SGD (co/khong momentum) va keras.optimizers.Adam
    # if name == "gd":
    #     ...
    # elif name == "momentum":
    #     ...
    # elif name == "adam":
    #     ...
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE

In [ ]:
#@title Answer { display-mode: "form" }
def get_optimizer(name, learning_rate=0.0007):
    '''
    Tra ve mot optimizer Keras tuong ung voi 3 thuat toan da tu cai o bai 04.

    Arguments:
    name -- "gd" | "momentum" | "adam"

    Returns: mot tf.keras.optimizers.Optimizer
    '''
    # (~6 dong) dung keras.optimizers.SGD (co/khong momentum) va keras.optimizers.Adam
    # YOUR CODE STARTS HERE
    if name == "gd":
        return keras.optimizers.SGD(learning_rate=learning_rate)
    elif name == "momentum":
        return keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    elif name == "adam":
        return keras.optimizers.Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.999)
    else:
        raise ValueError(f"Unknown optimizer name: {name}")
    # YOUR CODE ENDS HERE

## 3. Huan luyen & so sanh 3 optimizer

Dung `learning_rate=0.0007` va `batch_size=64` cho ca 3 (giong gia tri mac dinh cua `model()` o
bai 04). Ban NumPy dung `num_epochs=5000`; o day giam con `epochs=200` de notebook chay nhanh khi
minh hoa (van du de thay ro su khac biet giua GD/Momentum/Adam tren tap du lieu nho nay).

In [ ]:
EPOCHS = 200
histories = {}
for name in ["gd", "momentum", "adam"]:
    tf.random.set_seed(3)
    model = build_model(train_X.shape[0])
    optimizer = get_optimizer(name, learning_rate=0.0007)
    history = compile_and_train(model, train_X, train_Y, optimizer, epochs=EPOCHS, batch_size=64, verbose=0)
    histories[name] = {"model": model, "history": history}
    print(f"{name:10s} -> cost cuoi cung (train) = {history.history['loss'][-1]:.4f}")

In [ ]:
plt.figure(figsize=(7, 4.5))
for name in ["gd", "momentum", "adam"]:
    plt.plot(histories[name]["history"].history["loss"], label=name)
plt.xlabel("epoch")
plt.ylabel("cost")
plt.title("Cost qua qua trinh train - so sanh optimizer")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
for name in ["gd", "momentum", "adam"]:
    evaluate_classification(histories[name]["model"], test_X, test_Y, f'optimizer = "{name}" (test set)')

## 4. Tong ket

- Sau 200 epoch (`learning_rate=0.0007`, batch_size=64), ket qua tren test set: **`gd`** 67.5%
  accuracy (Recall 25.9%), **`momentum`** 72.1% (Recall 38.9%), **`adam`** cao nhat voi 74.7%
  accuracy, Recall 46.3%, F1 0.562. Cost cuoi cung: gd 0.638, momentum 0.502, adam **0.450** -
  Adam va Momentum deu giam cost nhanh hon GD thuan ro ret.
- Diem dang chu y: ca 3 optimizer deu con cach xa hoi tu hoan toan sau chi 200 epoch (bai 04 NumPy
  dung toi 5000 epoch de dat ~72-79%) - nhung thu tu **Adam > Momentum > GD** ve ca cost lan Recall
  da the hien ro ngay trong ngan sach epoch nho nay. Adam ket hop ca momentum (trung binh dong bac 1)
  lan dieu chinh learning rate thich nghi theo tung tham so (trung binh dong bac 2, giong `v` va `s`
  ban tu cai trong `update_parameters_with_adam`) nen hoi tu nhanh nhat; Momentum thuan cung nhanh
  hon GD nho tich luy da theo huong gradient on dinh - dung ly thuyet da hoc o bai 04.
- `model.fit(..., batch_size=64)` chinh la `random_mini_batches(..., mini_batch_size=64)` ban tu
  viet o bai 04 - Keras tu chia mini-batch VA tu **shuffle lai du lieu moi epoch**, dung yeu cau
  ban da doc trong docstring cua `random_mini_batches`.
- Thu bai tap mo rong: tang `epochs` len 2000-5000 cho `gd`/`momentum` de xem chung co dan bat kip
  `adam` khong (giong ket qua ~72% da thay o bai 04 NumPy voi du 5000 epoch); hoac doi rieng
  `learning_rate` cho tung optimizer bang `get_optimizer(name, learning_rate=...)` ban vua viet.